# AlohaMini Interactive Walkthrough

## A Guided Exploration of the AlohaMini Bimanual Mobile Manipulation System

---

**Welcome!** This notebook is designed to help you explore and understand the AlohaMini codebase through hands-on experimentation. 

**What you'll learn:**
- How the AlohaMini repository is organized
- What the Robot abstraction provides
- How teleoperation works (sensor → control → motor)
- How autonomous execution works (observation → policy → action)
- How data is recorded and structured
- Where timing, async behavior, and control flow happen

**Note:** This notebook assumes **no hardware is connected**. We'll use mock objects and inspection to explore the system.

---

## Part 1: Repository Orientation

Let's start by understanding how the repository is organized and where key components live.

In [ ]:
# First, let's check our current working directory and navigate to the repo root
import os
import sys
from pathlib import Path

# Get the notebook's directory
notebook_dir = Path.cwd()
print(f"Current directory: {notebook_dir}")

# Add the src directory to Python path so we can import lerobot modules
repo_root = notebook_dir
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"\nRepository root: {repo_root}")
print(f"Source path added: {src_path}")

In [ ]:
# Let's explore the key directories in this repository
import subprocess

print("\n=== Key Directories ===")
key_dirs = [
    "examples/alohamini",
    "src/lerobot/robots/alohamini",
    "src/lerobot/teleoperators",
    "src/lerobot/motors",
    "src/lerobot/cameras",
    "src/lerobot/policies",
    "src/lerobot/datasets"
]

for dir_path in key_dirs:
    full_path = repo_root / dir_path
    if full_path.exists():
        # Count Python files
        py_files = list(full_path.glob("*.py"))
        print(f"\n📁 {dir_path}")
        print(f"   Python files: {len(py_files)}")
        if py_files:
            print(f"   Files: {', '.join([f.name for f in py_files[:5]])}")
            if len(py_files) > 5:
                print(f"   ... and {len(py_files) - 5} more")

### Understanding the Naming: AlohaMini vs LeKiwi

You'll see both names in the codebase:

- **AlohaMini**: The physical robot platform (bimanual mobile manipulator)
- **LeKiwi**: The software implementation class name

They refer to the same system. The code uses "lekiwi" as the internal name for classes and modules.

In [ ]:
# Let's look at the AlohaMini-specific files
alohamini_dir = repo_root / "examples" / "alohamini"
lekiwi_robot_dir = repo_root / "src" / "lerobot" / "robots" / "alohamini"

def _get_first_docstring_line(filepath):
    """Extract the first meaningful line from a Python file"""
    try:
        with open(filepath) as f:
            lines = f.readlines()
            for line in lines[20:40]:  # Skip copyright header
                if 'class ' in line:
                    return line.split('class ')[1].split('(')[0].split(':')[0]
                if 'def ' in line and 'main' in line:
                    return "Main script"
    except:
        pass
    return "(implementation file)"

print("=== AlohaMini Example Scripts ===")
for f in sorted(alohamini_dir.glob("*.py")):
    print(f"  {f.name:40s} - {_get_first_docstring_line(f)}")

print("\n=== LeKiwi Robot Implementation ===")
for f in sorted(lekiwi_robot_dir.glob("*.py")):
    print(f"  {f.name:40s} - {_get_first_docstring_line(f)}")

---

## Part 2: Robot Abstraction and API

The core of AlohaMini is the `LeKiwi` robot class. Let's explore what the Robot API provides and how it abstracts hardware.

In [ ]:
# Install missing dependency
# %pip install draccus serial deepdiff numpy torch
# %pip install accelerate datasets opencv-python

# Import the LeKiwi robot classes
from lerobot.robots.alohamini.config_lekiwi import LeKiwiConfig, LeKiwiClientConfig
from lerobot.robots.alohamini.lekiwi import LeKiwi
from lerobot.robots.alohamini.lekiwi_client import LeKiwiClient

print("✓ Successfully imported LeKiwi robot classes")
print(f"  - LeKiwi: {LeKiwi}")
print(f"  - LeKiwiClient: {LeKiwiClient}")
print(f"  - LeKiwiConfig: {LeKiwiConfig}")

### Exploring Robot Configuration

Before we can create a robot, we need to understand its configuration. The config defines:
- Serial ports for motor buses
- Safety limits
- Camera settings
- Motor normalization modes

In [ ]:
# Create a default configuration
config = LeKiwiConfig()

print("=== LeKiwi Default Configuration ===")
print(f"Left motor bus port:  {config.left_port}")
print(f"Right motor bus port: {config.right_port}")
print(f"Max relative target:  {config.max_relative_target}")
print(f"Use degrees:          {config.use_degrees}")
print(f"Cameras configured:   {len(config.cameras)}")
print(f"Disable torque on disconnect: {config.disable_torque_on_disconnect}")

# Let's see what cameras are configured (if any)
if config.cameras:
    print("\nConfigured cameras:")
    for cam_name, cam_config in config.cameras.items():
        print(f"  - {cam_name}: {cam_config}")
else:
    print("\n⚠️  No cameras configured (this is normal in default config)")

### Understanding Observation and Action Spaces

The Robot API provides two key properties:
- **`observation_features`**: What the robot can sense (joint positions, velocities, camera images)
- **`action_features`**: What we can command (target positions, velocities)

Let's inspect these without connecting to hardware.

In [ ]:
# We can't instantiate the robot without hardware, but we can inspect the class
# to understand its state features

# Create a mock robot instance for inspection (won't connect to hardware)
# We'll use the class's _state_ft property which is defined statically

import inspect

# Get the source code of the _state_ft property
print("=== Robot State Features (from source code) ===")
print("\nThe robot's state includes:")

state_features = [
    # Left arm
    "arm_left_shoulder_pan.pos",
    "arm_left_shoulder_lift.pos",
    "arm_left_elbow_flex.pos",
    "arm_left_wrist_flex.pos",
    "arm_left_wrist_roll.pos",
    "arm_left_gripper.pos",
    # Right arm
    "arm_right_shoulder_pan.pos",
    "arm_right_shoulder_lift.pos",
    "arm_right_elbow_flex.pos",
    "arm_right_wrist_flex.pos",
    "arm_right_wrist_roll.pos",
    "arm_right_gripper.pos",
    # Base (body-frame velocities)
    "x.vel",
    "y.vel",
    "theta.vel",
    # Lift axis
    "lift_axis.height_mm",
]

print("\n📊 Arm Features (12 DOF total):")
for feature in state_features[:12]:
    print(f"  {feature}")

print("\n🚗 Base Features (omnidirectional):")
for feature in state_features[12:15]:
    print(f"  {feature}")

print("\n↕️  Lift Axis:")
print(f"  {state_features[15]}")

print(f"\n📷 Camera Features: (configurable, currently 0)")
print("\nTotal state dimensions: {} (+ camera images)".format(len(state_features)))

### Creating a Mock Observation Dictionary

Let's create a sample observation to understand the data structure that flows through the system.

In [ ]:
import numpy as np

# Create a mock observation dictionary (what robot.get_observation() would return)
mock_observation = {
    # Left arm positions (normalized -100 to +100 in default mode)
    "arm_left_shoulder_pan.pos": 0.0,
    "arm_left_shoulder_lift.pos": -30.0,
    "arm_left_elbow_flex.pos": 45.0,
    "arm_left_wrist_flex.pos": -15.0,
    "arm_left_wrist_roll.pos": 0.0,
    "arm_left_gripper.pos": 50.0,  # 0=open, 100=closed
    
    # Right arm positions
    "arm_right_shoulder_pan.pos": 0.0,
    "arm_right_shoulder_lift.pos": -30.0,
    "arm_right_elbow_flex.pos": 45.0,
    "arm_right_wrist_flex.pos": -15.0,
    "arm_right_wrist_roll.pos": 0.0,
    "arm_right_gripper.pos": 50.0,
    
    # Base velocities (body frame)
    "x.vel": 0.0,      # m/s (forward/backward)
    "y.vel": 0.0,      # m/s (left/right)
    "theta.vel": 0.0,  # deg/s (rotation)
    
    # Lift axis
    "lift_axis.height_mm": 150.0,  # mm (0-600 range)
    
    # Cameras (if configured) - would be numpy arrays of shape (H, W, 3)
    # "head_top": np.zeros((480, 640, 3), dtype=np.uint8),
}

print("=== Mock Observation Dictionary ===")
print(f"\nTotal keys: {len(mock_observation)}")
print("\nSample values:")
for key, value in list(mock_observation.items())[:5]:
    print(f"  {key:35s} = {value:8.2f}")
print("  ...")

# This is the data structure that flows through the system:
# Robot → Observation → Policy → Action → Robot
print("\n✓ This dictionary structure is what policies receive as input")

---

## Part 3: Teleoperation Execution Path

Now let's explore how teleoperation works. We'll walk through the control loop that merges inputs from:
1. Leader arms (physical manipulation)
2. Keyboard (base control)
3. Voice commands (advanced control)

And sends them to the robot.

In [ ]:
# Let's examine the teleoperation script structure
teleop_script = repo_root / "examples" / "alohamini" / "teleoperate_bi_voice.py"

print("=== Teleoperation Script Structure ===")
print(f"Script: {teleop_script.name}\n")

# Read and analyze the main loop
with open(teleop_script) as f:
    lines = f.readlines()

# Find the main loop (around line 77-105)
in_main_loop = False
main_loop_lines = []
for i, line in enumerate(lines[70:110], start=71):
    if 'while True:' in line or in_main_loop:
        in_main_loop = True
        main_loop_lines.append(f"{i:3d}: {line.rstrip()}")
        if 'precise_sleep' in line:
            break

print("Main control loop (simplified):")
for line in main_loop_lines[:15]:
    print(line)
print("...")

### Simulating the Teleoperation Loop

Let's simulate what happens in each iteration of the teleoperation loop.

In [ ]:
# Simulate one iteration of the teleoperation loop
import time

def simulate_teleop_iteration():
    """
    Simulates one iteration of the teleoperation control loop.
    This shows how actions from different sources are merged.
    """
    print("\n" + "="*60)
    print("TELEOPERATION ITERATION (30 Hz loop)")
    print("="*60)
    
    # Step 1: Get current observation from robot
    print("\n[1] Get current observation from robot")
    observation = mock_observation.copy()
    cur_height = observation["lift_axis.height_mm"]
    print(f"    Current lift height: {cur_height} mm")
    
    # Step 2: Poll voice engine
    print("\n[2] Poll voice engine for commands")
    # Mock: simulate voice command "forward 2 seconds"
    voice_text = None  # In real system: speech.get_text_nowait()
    if voice_text:
        print(f"    🎤 Voice command: '{voice_text}'")
    else:
        print("    (no voice command)")
    
    # Step 3: Get leader arm actions
    print("\n[3] Read leader arm positions")
    # Mock: simulate leader arms in neutral position
    arm_actions = {
        "arm_left_shoulder_pan.pos": 0.0,
        "arm_left_shoulder_lift.pos": -25.0,
        "arm_left_elbow_flex.pos": 40.0,
        "arm_left_wrist_flex.pos": -10.0,
        "arm_left_wrist_roll.pos": 0.0,
        "arm_left_gripper.pos": 60.0,
        # Right arm...
        "arm_right_shoulder_pan.pos": 0.0,
        "arm_right_shoulder_lift.pos": -25.0,
        "arm_right_elbow_flex.pos": 40.0,
        "arm_right_wrist_flex.pos": -10.0,
        "arm_right_wrist_roll.pos": 0.0,
        "arm_right_gripper.pos": 60.0,
    }
    print(f"    Leader positions: {len(arm_actions)} DOF")
    print(f"    Sample: left_gripper = {arm_actions['arm_left_gripper.pos']}")
    
    # Step 4: Get keyboard input
    print("\n[4] Process keyboard input")
    # Mock: simulate 'w' key pressed (forward)
    pressed_keys = ['w']  # In real system: keyboard.get_action()
    base_action = _mock_keyboard_to_base_action(pressed_keys)
    print(f"    Keys pressed: {pressed_keys}")
    print(f"    Base action: {base_action}")
    
    # Step 5: Get voice actions
    print("\n[5] Get voice executor actions")
    # Mock: no voice action currently
    voice_action = {}  # In real system: execu.get_action_nowait()
    print(f"    Voice action: {voice_action if voice_action else '(none)'}")
    
    # Step 6: Merge all actions
    print("\n[6] Merge all actions (later overwrites earlier)")
    merged_action = {**arm_actions, **base_action, **voice_action}
    print(f"    Total action keys: {len(merged_action)}")
    print(f"    Arm actions: {len(arm_actions)} keys")
    print(f"    Base actions: {len(base_action)} keys")
    print(f"    Voice actions: {len(voice_action)} keys")
    
    # Step 7: Send to robot
    print("\n[7] Send merged action to robot")
    print(f"    robot.send_action({list(merged_action.keys())[:3]}...)")
    print("    → ZMQ push to LeKiwiHost")
    print("    → Robot executes action")
    
    # Step 8: Timing
    print("\n[8] Maintain 30 Hz timing")
    loop_time_ms = 33.3  # 1/30 seconds
    print(f"    Target loop time: {loop_time_ms:.1f} ms")
    print(f"    precise_sleep() compensates for execution time")
    
    print("\n" + "="*60 + "\n")

def _mock_keyboard_to_base_action(keys):
    """Convert keyboard keys to base velocity commands"""
    action = {"x.vel": 0.0, "y.vel": 0.0, "theta.vel": 0.0}
    if 'w' in keys:
        action["x.vel"] = 0.2  # Forward at 0.2 m/s
    if 's' in keys:
        action["x.vel"] = -0.2  # Backward
    if 'a' in keys:
        action["theta.vel"] = 45.0  # Rotate left
    if 'd' in keys:
        action["theta.vel"] = -45.0  # Rotate right
    return action

# Run the simulation
simulate_teleop_iteration()

### Understanding Action Merging

A key insight: actions from different sources are merged using Python's `**` operator. Later dictionaries overwrite earlier ones.

In [ ]:
# Demonstrate action merging behavior
print("=== Action Merging Example ===")

# Source 1: Arm actions from leader
arm_actions = {
    "arm_left_gripper.pos": 50.0,
    "x.vel": 0.0,  # Arms don't control base, but key exists
}

# Source 2: Base actions from keyboard
keyboard_actions = {
    "x.vel": 0.2,  # Overwrites arm's x.vel
    "y.vel": 0.0,
}

# Source 3: Voice actions
voice_actions = {
    "x.vel": 0.0,  # STOP command - overwrites keyboard
    "y.vel": 0.0,
    "theta.vel": 0.0,
}

print("Source 1 (arms):    ", arm_actions)
print("Source 2 (keyboard):", keyboard_actions)
print("Source 3 (voice):   ", voice_actions)

# Merge: {**dict1, **dict2, **dict3}
merged = {**arm_actions, **keyboard_actions, **voice_actions}

print("\nMerged action:      ", merged)
print("\n💡 Notice: x.vel is 0.0 (from voice), not 0.2 (from keyboard)")
print("   Later sources have priority in the merge order!")

---

## Part 4: Autonomous Execution Path

Let's explore how a trained policy controls the robot autonomously.

**Flow:** Observation → Preprocessor → Policy → Postprocessor → Action

In [ ]:
# Install necessary packages
# %pip install pillow lerobot

# Let's examine what a policy interface looks like
from lerobot.policies.act.configuration_act import ACTConfig

print("=== ACT Policy Configuration ===")

# Create a default ACT policy config
act_config = ACTConfig()

print(f"\nInput shapes:")
print(f"  Chunk size (action horizon): {act_config.chunk_size}")
print(f"  Number of action steps: {act_config.n_action_steps}")

print(f"\nArchitecture:")
print(f"  Backbone: {act_config.vision_backbone}")
print(f"  Hidden dimension: {act_config.dim_model}")
print(f"  Feedforward dimension: {act_config.dim_feedforward}")
print(f"  Number of encoder layers: {act_config.n_encoder_layers}")
print(f"  Number of decoder layers: {act_config.n_decoder_layers}")

print("\n💡 ACT uses a transformer-based architecture to predict future action sequences")

### Creating a Dummy Policy for Demonstration

Since we don't have a trained model, let's create a simple scripted "policy" to understand the flow.

In [ ]:
class DummyPolicy:
    """
    A simple scripted policy for demonstration.
    
    This shows the interface that all policies must implement:
    - select_action(observation) -> action
    """
    def __init__(self):
        self.step_count = 0
        
    def select_action(self, observation):
        """
        Given an observation, return an action.
        
        In a real policy, this would:
        1. Encode observations (images → features)
        2. Run transformer decoder
        3. Decode to action space
        """
        self.step_count += 1
        
        # Simple behavior: slowly close grippers, move base forward
        action = observation.copy()  # Start with current state
        
        # Modify some values
        action["arm_left_gripper.pos"] = min(100.0, 50.0 + self.step_count * 2)
        action["arm_right_gripper.pos"] = min(100.0, 50.0 + self.step_count * 2)
        action["x.vel"] = 0.1  # Move forward slowly
        
        return action

# Create and test the dummy policy
policy = DummyPolicy()

print("=== Dummy Policy Execution ===")
print("\nRunning policy for 5 steps:\n")

current_obs = mock_observation.copy()

for step in range(5):
    print(f"Step {step + 1}:")
    print(f"  Input obs: gripper={current_obs['arm_left_gripper.pos']:.1f}, height={current_obs['lift_axis.height_mm']:.1f}mm")
    
    # Policy predicts action
    action = policy.select_action(current_obs)
    
    print(f"  Output action: gripper={action['arm_left_gripper.pos']:.1f}, x.vel={action['x.vel']:.2f}m/s")
    
    # Simulate robot executing action (update observation)
    current_obs['arm_left_gripper.pos'] = action['arm_left_gripper.pos']
    current_obs['arm_right_gripper.pos'] = action['arm_right_gripper.pos']
    print()

print("✓ This demonstrates the observation → action loop that policies use")

### Understanding Preprocessors and Postprocessors

Real policies need data normalization. Let's see how that works.

In [ ]:
# Simulate normalization preprocessing
import numpy as np

class SimpleNormalizer:
    """
    Demonstrates how observations are normalized before going to the policy.
    Real implementation is in lerobot.processor.normalize_processor
    """
    def __init__(self, stats):
        self.mean = stats['mean']
        self.std = stats['std']
    
    def normalize(self, value):
        return (value - self.mean) / self.std
    
    def denormalize(self, normalized_value):
        return normalized_value * self.std + self.mean

# Example: gripper position normalization
# Computed from training dataset statistics
gripper_stats = {
    'mean': 50.0,  # Average gripper position in dataset
    'std': 20.0,   # Standard deviation
}

normalizer = SimpleNormalizer(gripper_stats)

print("=== Normalization Example ===")
print(f"\nDataset statistics:")
print(f"  Mean: {gripper_stats['mean']}")
print(f"  Std:  {gripper_stats['std']}")

raw_values = [0.0, 25.0, 50.0, 75.0, 100.0]
print("\nNormalization:")
for val in raw_values:
    normalized = normalizer.normalize(val)
    print(f"  {val:5.1f} → {normalized:6.2f} (normalized)")

print("\nDenormalization (what policy outputs):")
normalized_values = [-2.5, 0.0, 1.0]
for val in normalized_values:
    denormalized = normalizer.denormalize(val)
    print(f"  {val:5.2f} → {denormalized:6.1f} (denormalized)")

print("\n💡 Normalization helps neural networks train more effectively")
print("   Values centered around 0 with std=1 are easier to learn")

---

## Part 5: Sensors and Actuation

Let's explore the hardware abstraction layer: motors, cameras, and the lift axis.

In [ ]:
# Explore motor configuration
from lerobot.motors import Motor, MotorNormMode

print("=== Motor Configuration ===")

# Example: Define a motor like in LeKiwi
shoulder_pan = Motor(
    id=1,
    model="sts3215",
    norm_mode=MotorNormMode.RANGE_M100_100
)

print(f"\nMotor: Shoulder Pan")
print(f"  ID: {shoulder_pan.id}")
print(f"  Model: {shoulder_pan.model}")
print(f"  Normalization mode: {shoulder_pan.norm_mode}")

print("\n=== All Motor Buses in LeKiwi ===")
print("\nLeft Bus (/dev/am_arm_follower_left):")
left_bus_motors = [
    (1, "arm_left_shoulder_pan", "Position"),
    (2, "arm_left_shoulder_lift", "Position"),
    (3, "arm_left_elbow_flex", "Position"),
    (4, "arm_left_wrist_flex", "Position"),
    (5, "arm_left_wrist_roll", "Position"),
    (6, "arm_left_gripper", "Position"),
    (8, "base_left_wheel", "Velocity"),
    (9, "base_back_wheel", "Velocity"),
    (10, "base_right_wheel", "Velocity"),
    (11, "lift_axis", "Velocity"),
]

for motor_id, name, mode in left_bus_motors:
    print(f"  ID {motor_id:2d}: {name:25s} [{mode}]")

print("\nRight Bus (/dev/am_arm_follower_right):")
right_bus_motors = [
    (1, "arm_right_shoulder_pan", "Position"),
    (2, "arm_right_shoulder_lift", "Position"),
    (3, "arm_right_elbow_flex", "Position"),
    (4, "arm_right_wrist_flex", "Position"),
    (5, "arm_right_wrist_roll", "Position"),
    (6, "arm_right_gripper", "Position"),
]

for motor_id, name, mode in right_bus_motors:
    print(f"  ID {motor_id:2d}: {name:25s} [{mode}]")

print("\n💡 Notice:")
print("  - Arms use Position mode (target angle)")
print("  - Base wheels use Velocity mode (target speed)")
print("  - Lift axis uses Velocity mode with P-controller")

### Understanding Omnidirectional Base Kinematics

The base has 3 wheels arranged at 120° intervals. Let's see how body velocities convert to wheel commands.

In [ ]:
# Simplified version of LeKiwi._body_to_wheel_raw
import numpy as np

def body_to_wheel_velocities(x_vel, y_vel, theta_vel):
    """
    Convert body-frame velocities to wheel velocities.
    
    Args:
        x_vel: Forward velocity (m/s)
        y_vel: Leftward velocity (m/s)
        theta_vel: Rotation velocity (deg/s)
    
    Returns:
        Dictionary with wheel velocities
    """
    wheel_radius = 0.05  # 5cm wheels
    base_radius = 0.125  # 12.5cm from center to wheel
    
    # Convert theta to rad/s
    theta_rad = theta_vel * (np.pi / 180.0)
    
    # Body velocity vector
    velocity_vector = np.array([-x_vel, -y_vel, theta_rad])
    
    # Wheel angles (120° spacing with -90° offset)
    angles = np.radians([240, 0, 120]) - np.pi/2
    
    # Kinematic matrix
    M = np.array([
        [np.cos(angles[0]), np.sin(angles[0]), base_radius],
        [np.cos(angles[1]), np.sin(angles[1]), base_radius],
        [np.cos(angles[2]), np.sin(angles[2]), base_radius],
    ])
    
    # Compute wheel linear speeds
    wheel_linear_speeds = M @ velocity_vector
    
    # Convert to angular speeds (rad/s)
    wheel_angular_speeds = wheel_linear_speeds / wheel_radius
    
    return {
        'left_wheel': wheel_angular_speeds[0],
        'back_wheel': wheel_angular_speeds[1],
        'right_wheel': wheel_angular_speeds[2],
    }

# Test different motion commands
print("=== Omnidirectional Base Kinematics ===")

test_cases = [
    (0.2, 0.0, 0.0, "Forward"),
    (0.0, 0.2, 0.0, "Left strafe"),
    (0.0, 0.0, 45.0, "Rotate CCW"),
    (0.2, 0.1, 0.0, "Forward + Left"),
]

for x, y, theta, description in test_cases:
    wheels = body_to_wheel_velocities(x, y, theta)
    print(f"\n{description}:")
    print(f"  Body command: x={x:.2f} m/s, y={y:.2f} m/s, θ={theta:.1f} deg/s")
    print(f"  Wheel speeds (rad/s):")
    print(f"    Left:  {wheels['left_wheel']:6.2f}")
    print(f"    Back:  {wheels['back_wheel']:6.2f}")
    print(f"    Right: {wheels['right_wheel']:6.2f}")

print("\n💡 The robot can move in any direction without rotating!")
print("   This is the power of omnidirectional drive.")

### Exploring the Lift Axis Controller

The lift axis is special: it uses multi-turn tracking and a P-controller.

In [ ]:
# Simulate the lift axis P-controller
class SimplifiedLiftAxis:
    """
    Simplified version of the LiftAxis controller.
    Demonstrates multi-turn tracking and P-control.
    """
    def __init__(self):
        self.lead_screw_mm_per_rev = 84.0  # mm per motor revolution
        self.extended_ticks = 0  # Multi-turn tick accumulator
        self.z0_ticks = 0  # Zero reference after homing
        self.target_mm = None
        self.KP = 300  # Proportional gain
        self.v_max = 1300  # Max velocity
        
    def get_height_mm(self):
        """Convert ticks to millimeters"""
        ticks_from_zero = self.extended_ticks - self.z0_ticks
        degrees_from_zero = ticks_from_zero * 360.0 / 4096.0
        revolutions = degrees_from_zero / 360.0
        height_mm = revolutions * self.lead_screw_mm_per_rev
        return height_mm
    
    def set_target(self, target_mm):
        """Set target height"""
        self.target_mm = np.clip(target_mm, 0, 600)
    
    def update(self, current_ticks):
        """
        P-controller update (called every frame).
        Returns commanded velocity.
        """
        # Update multi-turn tracking
        self.extended_ticks = current_ticks
        
        if self.target_mm is None:
            return 0  # No target, no motion
        
        # Compute error
        current_mm = self.get_height_mm()
        error_mm = self.target_mm - current_mm
        
        # Check if reached
        if abs(error_mm) < 1.0:  # 1mm tolerance
            self.target_mm = None
            return 0
        
        # P-control
        velocity = self.KP * error_mm
        velocity = np.clip(velocity, -self.v_max, self.v_max)
        
        return velocity

# Simulate lift axis motion
print("=== Lift Axis P-Controller Simulation ===")

lift = SimplifiedLiftAxis()
lift.z0_ticks = 0  # Assume we're already homed

# Command: go to 200mm
lift.set_target(200.0)

print("\nTarget: 200mm")
print("\nSimulating controller updates:\n")

# Simulate position feedback (motor moves upward)
simulated_ticks = 0
for step in range(100):
    current_height = lift.get_height_mm()
    velocity = lift.update(simulated_ticks)
    
    print(f"Step {step:2d}: height={current_height:6.1f}mm, error={lift.target_mm - current_height if lift.target_mm else 0:6.1f}mm, vel={velocity:7.7f}")
    
    # Simulate motor moving (velocity → position)
    # Simplified: assume velocity directly affects position
    simulated_ticks += int(velocity * 0.1)  # Timestep effect
    
    if lift.target_mm is None:
        print("\n✓ Target reached!")
        break

print("\n💡 The P-controller gradually reduces velocity as it approaches the target")

---

## Part 6: Data Recording and Datasets

Let's explore how teleoperation data is recorded and structured for training.

In [ ]:
# Create a mock dataset structure
print("=== LeRobot Dataset Structure ===")

# Simulate what a dataset episode looks like
episode_length = 1800  # 60 seconds at 30 Hz
num_episodes = 50

print(f"\nExample dataset:")
print(f"  Name: username/my_pick_and_place")
print(f"  Episodes: {num_episodes}")
print(f"  Frames per episode: {episode_length}")
print(f"  Total frames: {num_episodes * episode_length:,}")
print(f"  Frequency: 30 Hz")

print("\nDirectory structure:")
print("""
username/my_pick_and_place/
├── README.md                          # Dataset card
├── meta/
│   ├── episode_data_index.parquet     # Episode metadata
│   └── stats.parquet                  # Normalization statistics
├── data/
│   ├── frame_000000.safetensors       # Episode 0, frame 0
│   ├── frame_000001.safetensors       # Episode 0, frame 1
│   ├── ...
│   ├── frame_001799.safetensors       # Episode 0, frame 1799
│   ├── frame_001800.safetensors       # Episode 1, frame 0
│   └── ...
└── videos/                            # Encoded camera videos
    ├── head_top_episode_000000.mp4
    ├── wrist_left_episode_000000.mp4
    └── ...
""")

print("💡 Each frame is stored as a separate .safetensors file")
print("   This allows efficient random access during training")

In [ ]:
# Simulate what a single frame contains
print("=== Single Frame Structure ===")

# Create a mock frame
frame = {
    # Metadata
    'episode_index': 0,
    'frame_index': 42,
    'timestamp': 1.4,  # 42 / 30 Hz
    
    # Observation: state
    'observation.state': np.array([0.0, -30.0, 45.0, -15.0, 0.0, 50.0,  # left arm
                                   0.0, -30.0, 45.0, -15.0, 0.0, 50.0,  # right arm
                                   0.0, 0.0, 0.0,                        # base vel
                                   150.0]),                              # lift height
    
    # Observation: individual keys (for convenience)
    'observation.arm_left_shoulder_pan.pos': 0.0,
    'observation.arm_left_shoulder_lift.pos': -30.0,
    # ... (all other observation keys)
    
    # Observation: cameras (if configured)
    # 'observation.head_top': np.zeros((480, 640, 3), dtype=np.uint8),
    
    # Action: same structure as observation
    'action': np.array([0.0, -25.0, 40.0, -10.0, 0.0, 60.0,  # left arm target
                        0.0, -25.0, 40.0, -10.0, 0.0, 60.0,  # right arm target
                        0.2, 0.0, 0.0,                        # base vel target
                        150.0]),                              # lift target
    
    'action.arm_left_shoulder_pan.pos': 0.0,
    'action.arm_left_shoulder_lift.pos': -25.0,
    # ... (all other action keys)
}

print(f"Frame {frame['frame_index']} from episode {frame['episode_index']}")
print(f"Timestamp: {frame['timestamp']:.2f}s\n")

print("Observation state shape:", frame['observation.state'].shape)
print("Action shape:", frame['action'].shape)

print("\nObservation sample:")
print(f"  Left gripper: {frame['observation.arm_left_shoulder_lift.pos']:.1f}")

print("\nAction sample:")
print(f"  Left gripper target: {frame['action.arm_left_shoulder_lift.pos']:.1f}")

print("\n💡 During training, these (observation, action) pairs teach the policy")
print("   Policy learns to map: observation → action")

In [ ]:
# Simulate normalization statistics computation
print("=== Dataset Normalization Statistics ===")

# Mock statistics (would be computed from entire dataset)
stats = {
    'observation.state': {
        'mean': np.array([0.0] * 16),  # Mean per dimension
        'std': np.array([30.0, 30.0, 30.0, 30.0, 30.0, 20.0,  # Left arm
                        30.0, 30.0, 30.0, 30.0, 30.0, 20.0,   # Right arm
                        0.3, 0.3, 60.0,                        # Base
                        100.0]),                               # Lift
        'min': np.array([-100.0] * 12 + [-0.5, -0.5, -90.0, 0.0]),
        'max': np.array([100.0] * 12 + [0.5, 0.5, 90.0, 600.0]),
    },
    'action': {
        'mean': np.array([0.0] * 16),
        'std': np.array([30.0, 30.0, 30.0, 30.0, 30.0, 20.0,
                        30.0, 30.0, 30.0, 30.0, 30.0, 20.0,
                        0.3, 0.3, 60.0,
                        100.0]),
        'min': np.array([-100.0] * 12 + [-0.5, -0.5, -90.0, 0.0]),
        'max': np.array([100.0] * 12 + [0.5, 0.5, 90.0, 600.0]),
    },
}

print("\nObservation statistics (first 6 dims - left arm):")
print(f"  Mean: {stats['observation.state']['mean'][:6]}")
print(f"  Std:  {stats['observation.state']['std'][:6]}")
print(f"  Min:  {stats['observation.state']['min'][:6]}")
print(f"  Max:  {stats['observation.state']['max'][:6]}")

print("\nBase velocity statistics:")
print(f"  Mean: {stats['observation.state']['mean'][12:15]}")
print(f"  Std:  {stats['observation.state']['std'][12:15]}")

print("\n💡 These statistics are used to normalize data during training")
print("   and denormalize policy outputs during execution")

---

## Part 7: Control Flow, Timing, and Async Behavior

Let's explore the timing constraints and asynchronous components in the system.

In [ ]:
# Demonstrate precise timing control
import time

def precise_sleep(duration_s):
    """
    Sleep for exactly duration_s seconds.
    Used to maintain constant loop frequency.
    """
    if duration_s > 0:
        time.sleep(duration_s)

print("=== Control Loop Timing ===")

target_hz = 30
target_loop_time = 1.0 / target_hz

print(f"\nTarget frequency: {target_hz} Hz")
print(f"Target loop time: {target_loop_time*1000:.1f} ms\n")

# Simulate 5 iterations with varying execution times
print("Simulating control loop with compensation:\n")

execution_times = [0.010, 0.025, 0.015, 0.030, 0.020]  # Mock execution times

for i, exec_time in enumerate(execution_times):
    loop_start = time.perf_counter()
    
    # Simulate work
    time.sleep(exec_time)
    
    # Compute sleep time
    elapsed = time.perf_counter() - loop_start
    sleep_time = max(target_loop_time - elapsed, 0)
    
    print(f"Iteration {i+1}:")
    print(f"  Work time:  {elapsed*1000:5.1f} ms")
    print(f"  Sleep time: {sleep_time*1000:5.1f} ms")
    print(f"  Total:      {(elapsed + sleep_time)*1000:5.1f} ms")
    
    precise_sleep(sleep_time)

print("\n💡 precise_sleep() ensures constant loop frequency")
print("   regardless of varying execution times")

In [ ]:
# Explore asynchronous components
print("=== Asynchronous Components in AlohaMini ===")

async_components = [
    {
        'name': 'Camera Reading',
        'location': 'src/lerobot/cameras/camera.py',
        'mechanism': 'Background thread reads frames continuously',
        'interface': 'async_read() returns latest frame without blocking',
        'why': 'Prevents main loop from waiting for camera I/O'
    },
    {
        'name': 'Voice ASR',
        'location': 'examples/alohamini/voice_engine_gummy.py',
        'mechanism': 'Daemon thread processes audio and runs ASR',
        'interface': 'get_text_nowait() polls queue for results',
        'why': 'ASR (DashScope API) is slow, cannot block control loop'
    },
    {
        'name': 'ZMQ Communication',
        'location': 'src/lerobot/robots/alohamini/lekiwi_client.py',
        'mechanism': 'Non-blocking sockets with NOBLOCK flag',
        'interface': 'recv_string(zmq.NOBLOCK) raises zmq.Again if no data',
        'why': 'Network I/O must not block control loop'
    },
]

for comp in async_components:
    print(f"\n📡 {comp['name']}")
    print(f"   Location:   {comp['location']}")
    print(f"   Mechanism:  {comp['mechanism']}")
    print(f"   Interface:  {comp['interface']}")
    print(f"   Why async:  {comp['why']}")

print("\n💡 Asynchronous components prevent the 30 Hz control loop from blocking")
print("   All I/O that could take >33ms runs in background threads or uses non-blocking calls")

### Visualizing the Complete System Loop

Let's put it all together and trace a complete iteration through the system.

In [ ]:
# Complete system trace
print("=== Complete System Trace (One Control Loop Iteration) ===")
print("""                                                                      
┌──────────────────────────────────────────────────────────────────┐
│ LAPTOP (Client)                                      Time: 0 ms  │
└──────────────────────────────────────────────────────────────────┘
  │
  ├─[1]─► Poll voice engine (non-blocking)               +0.1 ms
  │        speech.get_text_nowait() → check queue
  │
  ├─[2]─► Read leader arms (USB serial)                  +3 ms
  │        BiSOLeader.get_action() → sync read
  │
  ├─[3]─► Check keyboard (non-blocking)                  +0.1 ms
  │        KeyboardTeleop.get_action() → stdin
  │
  ├─[4]─► Merge actions from all sources                 +0.1 ms
  │        {**arm_act, **kbd_act, **voice_act}
  │
  ├─[5]─► Send action via ZMQ (non-blocking)             +0.5 ms
  │        LeKiwiClient.send_action() → ZMQ push
  │        └─► JSON serialization
  │
  ▼ Network (TCP)

┌──────────────────────────────────────────────────────────────────┐
│ ROBOT (AlohaMini)                                    Time: 4 ms  │
└──────────────────────────────────────────────────────────────────┘
  │
  ├─[6]─► LeKiwiHost receives action                     +0.5 ms
  │        zmq_cmd_socket.recv_string(NOBLOCK)
  │        └─► JSON deserialization
  │
  ├─[7]─► Reset watchdog timer                           +0.1 ms
  │        last_cmd_time = time.time()
  │
  ├─[8]─► LeKiwi.send_action()                           +8 ms
  │        ├─► Split left/right arm actions
  │        ├─► Convert base vel to wheel commands
  │        ├─► Safety check (max_relative_target)
  │        ├─► left_bus.sync_write("Goal_Position")
  │        ├─► right_bus.sync_write("Goal_Position")
  │        └─► left_bus.sync_write("Goal_Velocity")
  │
  ├─[9]─► LiftAxis.update() (P-controller)               +1 ms
  │        └─► Compute velocity from error
  │
  ├─[10]─► LeKiwi.get_observation()                      +10 ms
  │        ├─► left_bus.sync_read("Present_Position")
  │        ├─► right_bus.sync_read("Present_Position")
  │        ├─► left_bus.sync_read("Present_Velocity")
  │        ├─► Convert wheel vel to body vel
  │        ├─► Check overcurrent protection
  │        └─► camera.async_read() (non-blocking)
  │
  ├─[11]─► Encode images to base64                       +5 ms
  │        └─► cv2.imencode(".jpg", img)
  │
  ├─[12]─► Send observation via ZMQ                      +0.5 ms
  │        └─► zmq_observation_socket.send_string()
  │
  ▼ Network (TCP)

┌──────────────────────────────────────────────────────────────────┐
│ LAPTOP (Client)                                     Time: 29 ms  │
└──────────────────────────────────────────────────────────────────┘
  │
  ├─[13]─► Receive observation (non-blocking)            +0.5 ms
  │        LeKiwiClient.get_observation()
  │        └─► Decode base64 images
  │
  ├─[14]─► precise_sleep()                               +3.0 ms
  │        └─► Sleep remainder to hit 33.3ms target
  │
  └─► End of iteration                                  Total: 33ms
      Loop back to [1] for next iteration
""")

print("\n💡 Total loop time: ~33ms = 30 Hz")
print("   Critical sections: motor communication (~18ms)")
print("   Async components prevent blocking: cameras, voice, network")

---

## Part 8: Key Takeaways and Navigation Guide

Congratulations! You've explored the AlohaMini codebase. Here's a summary of what you learned.

In [ ]:
print("""
═══════════════════════════════════════════════════════════════════
                     🎓 Key Takeaways
═══════════════════════════════════════════════════════════════════

1. ARCHITECTURE
   • AlohaMini = bimanual mobile manipulator (12-DOF arms + base + lift)
   • LeKiwi = software implementation (client-server via ZMQ)
   • Two motor buses: left (arm + base + lift), right (arm only)

2. TELEOPERATION
   • Multiple input sources: leader arms, keyboard, voice
   • Actions merged with dict unpacking: {**arm, **kbd, **voice}
   • 30 Hz control loop with precise timing compensation

3. AUTONOMOUS EXECUTION
   • Observation → Preprocessor → Policy → Postprocessor → Action
   • Normalization using dataset statistics (mean, std)
   • Policy predicts action sequences (chunk_size future steps)

4. SENSORS & ACTUATION
   • Arms: Position mode (target angle)
   • Base: Velocity mode (omnidirectional kinematics)
   • Lift: Velocity mode with P-controller and multi-turn tracking
   • Cameras: Async reading to prevent blocking

5. DATA & TRAINING
   • Dataset = (observation, action) pairs from teleoperation
   • Stored as .safetensors files on HuggingFace Hub
   • Statistics computed for normalization during training

6. TIMING & ASYNC
   • 30 Hz main loop (33.3ms per iteration)
   • Async components: cameras, voice ASR, ZMQ
   • Watchdog stops base if no command for >1.5s

═══════════════════════════════════════════════════════════════════
                  📚 Where to Go Next
═══════════════════════════════════════════════════════════════════

To dive deeper, explore these files:

🤖 Robot Implementation:
   src/lerobot/robots/alohamini/lekiwi.py
   src/lerobot/robots/alohamini/lekiwi_client.py
   src/lerobot/robots/alohamini/lift_axis.py

🎮 Teleoperation:
   examples/alohamini/teleoperate_bi_voice.py
   examples/alohamini/voice_engine_gummy.py
   examples/alohamini/voice_exec.py

🧠 Policies:
   src/lerobot/policies/act/modeling_act.py
   src/lerobot/policies/factory.py

💾 Datasets:
   src/lerobot/datasets/lerobot_dataset.py
   examples/alohamini/record_bi.py

⚙️ Motors & Cameras:
   src/lerobot/motors/feetech/feetech.py
   src/lerobot/cameras/opencv/camera_opencv.py

For comprehensive documentation, see:
   wiscohumanoid/docs/ALOHAMINI_ARCHITECTURE.md

═══════════════════════════════════════════════════════════════════
""")

print("\n✨ You now have the foundation to navigate and understand AlohaMini!")
print("   Try reading the source files with this context in mind.\n")

---

## Bonus: Quick Reference

Here are some useful code patterns you'll see throughout the codebase.

In [ ]:
print("""
═══════════════════════════════════════════════════════════════════
                  🔧 Common Code Patterns
═══════════════════════════════════════════════════════════════════

1. Filtering dictionary keys by prefix:
   left_pos = {k: v for k, v in action.items() 
               if k.endswith(".pos") and k.startswith("arm_left_")}

2. Merging actions from multiple sources:
   merged = {**source1, **source2, **source3}  # Later overwrites earlier

3. Non-blocking ZMQ receive:
   try:
       msg = socket.recv_string(zmq.NOBLOCK)
   except zmq.Again:
       # No message available
       pass

4. Maintaining constant loop frequency:
   loop_start = time.perf_counter()
   # ... do work ...
   elapsed = time.perf_counter() - loop_start
   time.sleep(max(1/fps - elapsed, 0))

5. Multi-turn tick tracking:
   delta = current_tick - last_tick
   if delta > 2048:
       delta -= 4096  # Wrapped backward
   elif delta < -2048:
       delta += 4096  # Wrapped forward
   extended_ticks += delta

6. Safe dictionary access with defaults:
   height = observation.get("lift_axis.height_mm", 0.0)

7. Normalizing data:
   normalized = (value - mean) / std
   denormalized = normalized * std + mean

8. Omnidirectional kinematics:
   angles = np.radians([240, 0, 120]) - np.pi/2
   M = [[cos(a), sin(a), base_radius] for a in angles]
   wheel_speeds = M @ body_velocity

═══════════════════════════════════════════════════════════════════
""")

---

## End of Walkthrough

Thank you for exploring the AlohaMini codebase! You now understand:

✅ How the repository is organized  
✅ What the Robot API provides  
✅ How teleoperation merges multiple input sources  
✅ How policies control the robot autonomously  
✅ How sensors and actuators are abstracted  
✅ How data is recorded for training  
✅ Where timing and async behavior happen  